# Day 06 Tutorial — Medallion Architecture & SCD Patterns

**Goal:** Bronze/silver/gold and SCD1; outline SCD2.


### Environment setup
Skip pip install on Databricks/Fabric. Locally you may need: `pip install pyspark pandas`.


In [ ]:
# %pip install pyspark==3.5.1 pandas -q


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('AzureDE-InterviewPrep')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark


## Medallion
- **Bronze:** raw append
- **Silver:** clean/dedupe
- **Gold:** business marts


In [ ]:
bronze = spark.createDataFrame(
    [
        (1, 'Alice', 'Pune', '2024-01-01 10:00:00'),
        (1, 'Alice', 'Mumbai', '2024-01-02 09:00:00'),
        (2, 'Bob', 'Delhi', '2024-01-01 11:00:00'),
        (2, 'Bob', 'Delhi', '2024-01-01 11:00:00'),
    ],
    ['customer_id', 'name', 'city', 'updated_at'],
)
w = Window.partitionBy('customer_id').orderBy(F.col('updated_at').desc())
silver = bronze.withColumn('rn', F.row_number().over(w)).filter('rn = 1').drop('rn')
silver.show()


## SCD Type 1


In [ ]:
dim = spark.createDataFrame(
    [(1, 'Alice', 'Pune'), (2, 'Bob', 'Delhi')],
    ['customer_id', 'name', 'city'],
)
scd1 = (
    dim.alias('d').join(silver.alias('s'), 'customer_id', 'left')
    .select(
        F.col('customer_id'),
        F.coalesce('s.name', 'd.name').alias('name'),
        F.coalesce('s.city', 'd.city').alias('city'),
    )
)
scd1.show()


## SCD2 outline
Use valid_from, valid_to, is_current. Close old row on change; insert new current row.

## Streaming concepts
Source -> transform -> sink + checkpoint; watermark for late data bounds.
